In [ ]:
!pip install -U transformers datasets openpyxl scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
#Imports
import os
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from google.colab import files

os.environ["TOKENIZERS_PARALLELISM"] = "false"

#Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

#Upload Excel Dataset
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name, engine="openpyxl")
df.columns = df.columns.str.strip()

print("Columns found:", df.columns)
print(df.head())

#Prepare Dataset
df = df.rename(columns={
    "comment": "text",
    "Comment": "text",
    "Label": "label",
    "Category": "label"
})

df = df[["text", "label"]].dropna()
df["label"] = df["label"].astype(int)

print("Label distribution:")
print(df["label"].value_counts())

#Train / Validation Split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_df = pd.DataFrame({"text": train_texts, "label": train_labels})
val_df = pd.DataFrame({"text": val_texts, "label": val_labels})

#Convert to HF Dataset
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

#Tokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")

print("Tokenization completed ✅")

#Load Model
model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=2
)

model.to(device)

#Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

#Training Arguments (FIXED VERSION)
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs"
)

#Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

#Train Model
trainer.train()

#Evaluate Model
results = trainer.evaluate()
print("Evaluation Results:", results)

#Save Model
trainer.save_model("cyberbullying_xlmr_model")
tokenizer.save_pretrained("cyberbullying_xlmr_model")

print("Model saved successfully ✅")

#Prediction Function
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=1).item()

    return "Cyberbullying" if prediction == 1 else "Non-Cyberbullying"





Using device: cuda


Saving multilingual_cyberbullying_10000_samples.xlsx to multilingual_cyberbullying_10000_samples.xlsx
Columns found: Index(['id', 'comment', 'label'], dtype='object')
   id             comment  label
0   1       මෙතනින් යන්න.      1
1   2       Good morning!      0
2   3  Oya pissu balliya.      1
3   4   Good work machan!      0
4   5       Shut up fool.      1
Label distribution:
label
1    5067
0    4933
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluation Results: {'eval_loss': 8.394563337787986e-06, 'eval_accuracy': 1.0, 'eval_f1': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_runtime': 13.9963, 'eval_samples_per_second': 142.895, 'eval_steps_per_second': 17.862, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully ✅


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the Excel file
data = pd.read_excel("multilingual_cyberbullying_10000_samples.xlsx")

# 2. Show columns (optional but useful)
print("Columns in dataset:", data.columns)

# 3. Define X (features) and y (target)
# This assumes the LAST column is the target
X = data.iloc[:, :-1]   # all columns except last
y = data.iloc[:, -1]    # last column

# 4. Split into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Check how much data was taken
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

total = len(X_train) + len(X_test)
print("Training %:", (len(X_train) / total) * 100)
print("Testing %:", (len(X_test) / total) * 100)


Columns in dataset: Index(['id', 'comment', 'label'], dtype='object')
Training samples: 8000
Testing samples: 2000
Training %: 80.0
Testing %: 20.0


In [ ]:
# Example test
print(predict("You are completely useless."))
print(predict("Nobody wants you here."))
print(predict("Shut up idiot."))
print(predict("Go disappear from this world."))
print(predict("You are such a failure."))

print(predict("ඔයා හරිම නිච්ච කෙනෙක්."))
print(predict("ඔයාට කිසිම වටිනාකමක් නැහැ."))
print(predict("කවුරුත් ඔයාව ඉවසන්නේ නැහැ."))
print(predict("මෙතනින් යන්න."))
print(predict("ඔයා හරිම වැරදි කෙනෙක්."))

print(predict("Oya harima useless kenek."))
print(predict("Umbata brain ekak nadda?"))
print(predict("Oya nikan pissu balliya."))
print(predict("Nobody like karanne oya."))
print(predict("Oya danne nehe wage inne."))

print(predict("Great job today!"))
print(predict("I really appreciate your help."))
print(predict("This post is amazing."))
print(predict("Happy birthday!"))
print(predict("Keep up the good work."))

print(predict("හොඳ වැඩක් කරලා තියෙනවා."))
print(predict("ඔයාගේ අදහස හරිම වටිනායි."))
print(predict("සුභ දවසක් වේවා!"))
print(predict("මේ post එක හරිම ලස්සනයි."))
print(predict("ඔයා හොඳ යාළුවෙක්."))

print(predict("Hariyata wada karala thiyenawa."))
print(predict("Mama me post ekata kemathi."))
print(predict("Oyage adahas hondai."))
print(predict("Good work machan!"))
print(predict("Suba dawasak wewa!"))

Cyberbullying
Cyberbullying
Cyberbullying
Cyberbullying
Cyberbullying
Cyberbullying
Cyberbullying
Cyberbullying
Cyberbullying
Cyberbullying
Cyberbullying
Cyberbullying
Cyberbullying
Cyberbullying
Cyberbullying
Non-Cyberbullying
Non-Cyberbullying
Non-Cyberbullying
Non-Cyberbullying
Non-Cyberbullying
Non-Cyberbullying
Non-Cyberbullying
Non-Cyberbullying
Non-Cyberbullying
Cyberbullying
Non-Cyberbullying
Non-Cyberbullying
Non-Cyberbullying
Non-Cyberbullying
Non-Cyberbullying


In [ ]:
import pickle

# Example: save your model
with open("Hate speech text detection model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully")

Model saved successfully


In [ ]:
from google.colab import files
files.download("Hate speech text detection model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>